Let's complete the integration process for all data.

For every folder, let's sum data, subtract the dark correction, and write a processed file as png and tiff. The png is viewable by cinema and the tiff will be integrated later. 

# Preprocessing and averaging of data

In [ ]:
from pathlib import Path
import numpy as np

# One experiment (fe2_ramp1, images 1-20) split by phase:
#   fe20_bccFe: images 1-6 (bcc), fe21_bcc_hcpCoexist: 7-9 (bcc + hcp), fe18_hcpFe: 10-20 (hcp)
SAMPLES = ["fe20_bccFe", "fe21_bcc_hcpCoexist", "fe18_hcpFe"]
TARGETS = [Path(s) / "data" for s in SAMPLES]
MASKINS = [target / "diff" / "mask.edf" for target in TARGETS]
MASKOUTS = [target / "diff" / "mask_full.edf" for target in TARGETS]
PONI_FILE = "calibration/detector.poni"
SEARCH_STR = "*-00???.tif"
GET_DIFF_FILES = lambda target: [Path(file) for file in list(Path(target / "diff").glob(SEARCH_STR))]
FILES = [sorted(GET_DIFF_FILES(target)) for target in TARGETS]
# PCO (sample camera) images: one set for the whole experiment, used for visualization only
PCO_DIR = Path("pco")
GET_PCO_KEY = lambda file: int(Path(file).stem.split("_")[-1])
FILES_PCO = [sorted(PCO_DIR.glob("fe2_ramp1_00001_*.tif"), key=GET_PCO_KEY)]
MASKIN_PCO = [PCO_DIR / "mask.edf"]
CHANGEEXT = lambda file,ext: f"{str(file)[:-4]}{ext}"
THRESHOLD_MAXS = [4096]*len(TARGETS)
THRESHOLD_MINS = [0]*len(TARGETS)
THRESHOLD_PCO_MAXS = [5000]
THRESHOLD_PCO_MINS = [0]
COLLECTION_TIMES = [0.1]*len(TARGETS)
WINDOW_AVERAGES = [2,5]

In [ ]:
#Make mask programatically 
from MILK.parallel import parallel_map

def build_mask(args):
    import fabio
    import numpy as np
    
    """Build a mask using thresholding and custom mask to start."""
    maskin, maskout,threshold_max,threshold_min,files = args
    maskedf = fabio.open(maskin)
    mask = maskedf.data
    for file in files:
        img = fabio.open(file)
        mask = mask | (img.data>threshold_max) | (img.data<threshold_min)
    maskedf.data = np.array(mask,dtype="uint8") 
    maskedf.write(maskout)
_ = parallel_map(build_mask, poolsize = 20, disable_tqdm=False, loop_arg = zip(MASKINS,MASKOUTS,THRESHOLD_MAXS,THRESHOLD_MINS,FILES))


In [ ]:

def process_images(maskout,file):
    import fabio
    import numpy as np

    CHANGEEXT = lambda file,ext: f"{str(file)[:-4]}{ext}"
    """Take a tif image, apply a mask, and save png plot and edf."""
    from pyFAI.gui import jupyter
    from matplotlib import pyplot as plt
    img = fabio.open(file)
    mask = fabio.open(maskout)
    ax = jupyter.display(np.ma.masked_array(img.data, mask.data))
    plt.savefig(CHANGEEXT(file,".png"))
    plt.close()
    img.write(CHANGEEXT(file,".edf"))

#Process_images in parallel
for files, maskout in zip(FILES,MASKOUTS):
    _ = parallel_map(process_images,20,False, files, maskout)

I started to do window averages below, but there wasn't a real need.

In [ ]:
# def process_window_avg_images(window):
#     """apply window averging to dataset."""
#     mask = fabio.open(MASKOUT)
#     for i in range(0,len(FILES)-window):
#         imgsum= fabio.open(CHANGEEXT(FILES[i],".edf")).data
#         for j in range(1,window):
#             img = fabio.open(CHANGEEXT(FILES[j],".edf"))
#             imgsum = img.data + imgsum
#         img.data = imgsum / window
#         ax = jupyter.display(np.ma.masked_array(img.data, mask.data))
#         plt.savefig(CHANGEEXT(FILES[i],f"_sum{window}.png"))
#         plt.close()
#         img.write(CHANGEEXT(FILES[i],f"_sum{window}.edf"))

# _ = parallel_map(process_window_avg_images,poolsize = 4,disable_tqdm=False, loop_arg = WINDOW_AVERAGES)

# #Do process_images in parallel
# for files, maskout in zip(FILES,MASKOUTS):
#     for window in WINDOW_AVERAGES
#         _ = parallel_map(process_images,6,False, files, maskout, window)

# Visualize Mask

Using a widget validate the various mask datasets.

In [ ]:
from ipywidgets import interact
import ipywidgets as widgets
import matplotlib.pyplot as plt
from PIL import Image

# Define the path to png images of interest
FILES = [sorted(GET_DIFF_FILES(target)) for target in TARGETS]
PNG_IMAGE = lambda sample,index: list((TARGETS[sample-1] / "diff").glob(f"*-{index:05d}.png"))
def plot(index,sample):
    """Plot function for the widget."""
    file = PNG_IMAGE(sample,index)
    if file != []:
        im = Image.open(file[0])
        plt.imshow(im);plt.axis('off')
    else:
        plt.imshow(np.zeros((1,1)));plt.axis('off')

wig_plot = interact(plot, index = widgets.IntSlider(value=1, min=1, max=max([len(files) for files in FILES]), sample=1), sample = widgets.IntSlider(value=1, min=1, max=len(FILES), sample=1))

In [ ]:
# Make cinema database 
import pandas as pd 
data = {'sample': [], 'run_id':[], 'FILE_0': []}

pd.DataFrame.from_dict(data)
for target, files in zip(TARGETS, FILES):
    data['FILE_0'] = data['FILE_0'] + [(Path("..") / CHANGEEXT(file,".png")).as_posix() for file in files]
    data['sample'] =  data['sample'] + [target.parts[0]]*len(files)
    data['run_id'] = data['run_id'] + list(range(1,len(files)+1))
df = pd.DataFrame.from_dict(data)
Path("diff_cinema_db").mkdir(exist_ok=True)
df.to_csv("diff_cinema_db/data.csv",index=False)

# To launch cinema use: milk-cinema -serve_path "./" -data_path "diff_cinema_db"

# PCO Processing

In [ ]:
def process_pco_images(maskin,threshold_min,threshold_max,file):
    from pyFAI.gui import jupyter
    from matplotlib import pyplot as plt
    from matplotlib.colors import SymLogNorm,TwoSlopeNorm,Normalize
    from pylab import subplots
    import fabio
    import numpy as np

    CHANGEEXT = lambda file,ext: f"{str(file)[:-4]}{ext}"
    """Take a tif image, plot with log scale and save as png."""
    
    img = fabio.open(file)
    mask = fabio.open(maskin)
    mask = (img.data > threshold_max) | mask.data | (img.data < threshold_min)
    data = np.ma.masked_array(img.data, mask.data)
    seg = np.where(data.mask==False)
    x_min = int(np.min(seg[1]))
    x_max = int(np.max(seg[1]))
    y_min = int(np.min(seg[0]))
    y_max = int(np.max(seg[0]))
    
    data = img.data[y_min:y_max,x_min:x_max]
    # print(np.nanmin(data.compressed()),np.nanmax(data.compressed()))
    # data = 1e6/np.nanmax(data-np.nanmin(data))*(data-np.nanmin(data))
    # print(data.compressed())
    # data = data
    colornorm = SymLogNorm(1, base=10,
                            vmin=np.nanmin(data),
                            vmax=np.nanmax(data))

    
    _fig, ax = subplots()
    ax.imshow(data,
              origin="lower",
              cmap="inferno",
              norm=colornorm)
    _fig.tight_layout()
    plt.savefig(CHANGEEXT(file,".png"))
    plt.close()

#Process_images in parallel
for files,maskin,threshold_pco_min,threshold_pco_max in zip(FILES_PCO,MASKIN_PCO,THRESHOLD_PCO_MINS,THRESHOLD_PCO_MAXS):
    _ = parallel_map(process_pco_images, 20, False, files, maskin,threshold_pco_min,threshold_pco_max)


In [ ]:
from ipywidgets import interact
import ipywidgets as widgets
import matplotlib.pyplot as plt
from PIL import Image

# Define the path to png images of interest
PNG_IMAGE = lambda sample,index: sorted(PCO_DIR.glob(f"*_{index:d}.png"), key=GET_PCO_KEY)
def plot(index,sample):
    """Plot function for the widget."""
    file = PNG_IMAGE(sample,index)
    if file != []:
        im = Image.open(file[0])
        plt.imshow(im);plt.axis('off')
    else:
        plt.imshow(np.zeros((1,1)));plt.axis('off')

wig_plot = interact(plot, index = widgets.IntSlider(value=1, min=1, max=len(FILES_PCO[0]), sample=1), sample = widgets.IntSlider(value=1, min=1, max=1, sample=1))

In [ ]:
# Make cinema database 
import pandas as pd 
data = {'sample': [], 'run_id':[], 'FILE_0': []}

pd.DataFrame.from_dict(data)
for target, files in zip([PCO_DIR], FILES_PCO):
    data['FILE_0'] = data['FILE_0'] + [(Path("..") / CHANGEEXT(file,".png")).as_posix() for file in files]
    data['sample'] =  data['sample'] + [target.parts[0]]*len(files)
    data['run_id'] = data['run_id'] + list(range(1,len(files)+1))
df = pd.DataFrame.from_dict(data)
Path("pco_cinema_db").mkdir(exist_ok=True)
df.to_csv("pco_cinema_db/data.csv",index=False)

# To launch cinema use: milk-cinema -serve_path "./" -data_path "pco_cinema_db"

## Integration
Using pyFAI integrate 1d and 2d. Make plots for visualizing.

In [ ]:
#Define the integration parameters except those we want to do programatically.
json_1d = {
    "poni_file": [],
    "do_mask": True,
    "mask_file": [],
    "do_dark": False,
    "dark_file": [],
    "dark_norm": 1.0,
    "do_bright": False,
    "bright_file": [],
    "bright_norm": 1.0,
    "data_ops": [],
    "do_polarization": False,
    "polarization_factor": 0.99,
    "do_2D": False,
    "unit": "2th_deg",
    "npt_radial": 2048,
    "do_radial_range": True,
    "radial_range": [8, 21],
    "npt_azimuth": 1,
    "do_azimuthal_range": False,
    "azimuth_range": [0.0, 360.0],
    "chi_discontinuity_at_0": True,
    "do_solid_angle": False,
    "do_remove_nan": True,
    "error_model": "Poisson",
    "method": [
        "full",
        "histogram",
        "cython"
    ],
    "opencl_device": "cpu"
}

json_2d = {
    "poni_file": [],
    "do_mask": True,
    "mask_file": [],
    "do_dark": False,
    "dark_file": [],
    "dark_norm": 1.0,
    "do_bright": False,
    "bright_file": [],
    "bright_norm": 1.0,
    "data_ops": [],
    "do_polarization": False,
    "polarization_factor": 0.99,
    "do_2D": False,
    "unit": "2th_deg",
    "npt_radial": 2048,
    "do_radial_range": True,
    "radial_range": [8, 21],
    "npt_azimuth": 72,
    "do_azimuthal_range": False,
    "azimuth_range": [0, 360],
    "chi_discontinuity_at_0": True,
    "do_solid_angle": False,
    "do_remove_nan": True,
    "error_model": "Poisson",
    "method": [
        "full",
        "histogram",
        "cython"
    ],
    "opencl_device": "cpu"
}

In [ ]:
#Setup function to write and update json files
import json
def update_json(j, **d):
    """Take json dictionary and replace items using keyword arguments **d."""
    for k, v in d.items():
        j[k] = v
    return j

def write_json(j, file):
    """Write json dictionary to file."""
    with open(file, "w") as f:
        f.write(json.dumps(j,indent=4))

In [ ]:
# Finish configuring jsons for each specimen
data_ops = [""]
bright_file = [""]
dark_file=[""]
do_dark = False
do_bright = False
poni_file = [PONI_FILE]

for files,maskout,target in zip(FILES,MASKOUTS,TARGETS):
    mask_file = [maskout.as_posix()]
    json_1d_file = target / "1d.azimint.json"
    json_2d_file = target / "2d.azimint.json"
    write_json(update_json(json_1d,
                            data_ops=data_ops,
                            bright_file=bright_file,
                            mask_file=mask_file,
                            do_dark=do_dark,
                            do_bright=do_bright,
                            poni_file=poni_file,
                            dark_file=dark_file), json_1d_file)
    write_json(update_json(json_2d,
                            data_ops=data_ops,
                            bright_file=bright_file,
                            mask_file=mask_file,
                            do_dark=do_dark,
                            do_bright=do_bright,
                            poni_file=poni_file,
                            dark_file=dark_file), json_2d_file)

In [ ]:
# Perform the integration 1D

def process_integration(target,file):
    from pathlib import Path
    from MILK.integration.integrate import main as int_main  
    CHANGEEXT = lambda file,ext: f"{str(file)[:-4]}{ext}"
    file_image=CHANGEEXT(file,'.edf')
    int_main(files=[file_image],
                json_file= target / "1d.azimint.json",
                output= Path(file_image).parent,
                overwrite=True,
                poolsize=1,
                formats=['esg'],
                histogram_plot=True,
                quiet=True,
                use_glob=True)

    # Perform the integration 2D 
    int_main(files=[file_image],
                json_file= target / "2d.azimint.json",
                output= Path(file_image).parent,
                overwrite=True,
                poolsize=1,
                formats=["esg1"],
                histogram_plot=True,
                quiet=True,
                use_glob=True)
for target,files in zip(TARGETS,FILES):
        _ = parallel_map(process_integration, 4, False, files, target)


In [ ]:
# Make cinema database 
import pandas as pd 
data = {'sample': [], 'run_id':[], 'FILE_0': [], 'FILE_1': [], 'FILE_2': []}

pd.DataFrame.from_dict(data)
for target, files in zip(TARGETS, FILES):
    data['FILE_0'] = data['FILE_0'] + [(Path("..") / CHANGEEXT(file,".png")).as_posix() for file in files]
    data['FILE_1'] = data['FILE_1'] + [(Path("..") / CHANGEEXT(file,"_1dplot.png")).as_posix() for file in files]
    data['FILE_2'] = data['FILE_2'] + [(Path("..") / CHANGEEXT(file,"_2dplot.png")).as_posix() for file in files]
    data['sample'] =  data['sample'] + [target.parts[0]]*len(files)
    data['run_id'] = data['run_id'] + list(range(1,len(files)+1))
df = pd.DataFrame.from_dict(data)
Path("diff_cinema_db").mkdir(exist_ok=True)
df.to_csv("diff_cinema_db/data.csv",index=True)

# To launch cinema use: milk-cinema -serve_path "./" -data_path "diff_cinema_db"

## Create cinema of complete dataset

In [ ]:
# Make cinema database 
# import pandas as pd 
# from os.path import getmtime

# df = pd.DataFrame()
# for target, files, files_pco, collection_time in zip(TARGETS, FILES, FILES_PCO,COLLECTION_TIMES):
#     data = {'sample': [],  'time':[]}#,'run_id':[], 'FILE_0': [], 'FILE_1': [], 'FILE_2': []}
#     data_pco = {'sample': [], 'time':[]}#, 'FILE_3': []}
    
#     data['FILE_0'] = [(Path("..") / CHANGEEXT(file,".png")).as_posix() for file in files]
#     data['FILE_1'] = [(Path("..") / CHANGEEXT(file,"_1dplot.png")).as_posix() for file in files]
#     data['FILE_2'] = [(Path("..") / CHANGEEXT(file,"_2dplot.png")).as_posix() for file in files]
#     data['sample'] = [target.parts[0]]*len(files)
#     # data['run_id'] = list(range(1,len(files)+1))
#     data['time'] = pd.to_datetime([getmtime(str(file)) for file in files])
#     df_diff = pd.DataFrame.from_dict(data)

#     data_pco['FILE_3'] = [(Path("..") / CHANGEEXT(file,".png")).as_posix() for file in files_pco]
#     data_pco['sample'] = [target.parts[0]]*len(files_pco)
#     data_pco["time"] = pd.to_datetime([getmtime(str(file)) for file in files_pco])
#     df_pco = pd.DataFrame.from_dict(data_pco)

#     df_a = pd.merge_asof(df_diff.sort_values("time"), df_pco[["time","sample","FILE_3"]].sort_values("time"), on="time", by="sample", tolerance=pd.Timedelta(f"{collection_time}s"))
#     df_a.head(30)
#     print("stop")
# df = pd.DataFrame.from_dict(data)
# Path("diff_cinema_db").mkdir(exist_ok=True)
df.to_csv("diff_cinema_db/data.csv",index=False)

# To launch cinema use: milk-cinema -serve_path "./" -data_path "diff_cinema_db"